# WV Quantile Embedding Benchmark

This notebook revises the WorldValue plug-in benchmark so it stays aligned with the calibrated curve pipeline in [WV_quantile_construction.ipynb](WV_quantile_construction.ipynb) while clearly separating theorem-backed calibrated curves from predictive plug-in references.

Top-level summary

- Reused workflow: retained 235-question subset, simulator bundle, `full` / `full_summary`, adaptive `gamma_j`, and calibrated pseudo-discrepancy construction are taken from the same pipeline used in [WV_quantile_construction.ipynb](WV_quantile_construction.ipynb).
- Human target source: the notebook searches the repo for the strongest retained-qid real-data source and prefers raw full human WVS responses, then full-data aggregates, then smaller-sample fallbacks.
- Target object: the predictive benchmark estimates the human-side target `p`, not the simulator side `q`. The discrepancy remains `loss(p_tilde, qhat)`.
- Predictive baselines:
  - `plugin_X`: question-only predictor `p_tilde_X = f(X)`
  - `plugin_XQ`: simulator-assisted predictor `p_tilde_XQ = f_sim(X, qhat)`
- Indexing:
  - calibrated curves keep the current notebook’s theorem-shifted quantile index
  - predictive curves are raw empirical quantile curves with no finite-sample index adjustment
- Embedding path:
  - it uses the repo's cached OpenAI `text-embedding-3-small` question embeddings when they are available for the retained qids


## SECTION 1 — Preparation

This section keeps the setup, repo discovery, data loading, real-data target selection, and question-feature construction in one place so the benchmark reads as a single preparation workflow.

Keep the object names and path conventions close to the current WorldValue quantile notebook so the benchmark is a natural extension rather than a separate workflow.


Embedding provenance

This notebook is fully runnable from the committed cached embedding artifact in the reproduction tree. For transparency, a commented reproduction snippet adapted from [WV_simulator_selection.ipynb](WV_simulator_selection.ipynb) is included below so readers can see how the cached question embeddings were originally generated.


In [ ]:
# Commented embedding-generation reference adapted from WV_simulator_selection.ipynb.
# This notebook does not execute the API call by default because the cached artifact is
# already committed at:
#   data/worldvalue/simulator_selector/wvs_qid_embeddings_openai_text-embedding-3-small.pkl
#
# To regenerate that artifact, uncomment and run a block like this in the same
# paper_reproduction/worldvalue_quantile environment.
#
# from openai import OpenAI
# import os
# import pickle
# import numpy as np
# import pandas as pd
# from pathlib import Path
#
# REPRO_ROOT = find_repo_root(Path.cwd())
# embed_model = "text-embedding-3-small"
# batch_size = 32
# api_key = os.getenv("OPENAI_API_KEY", "").strip()
# if not api_key:
#     raise RuntimeError("OPENAI_API_KEY is required to regenerate the cached embedding artifact.")
#
# client = OpenAI(api_key=api_key)
#
# def batched(seq, n):
#     for i in range(0, len(seq), n):
#         yield seq[i:i+n]
#
# qid_text_df = question_df[["qid", "question_text", "category", "question_representation"]].copy()
# qid_text_df = qid_text_df.rename(columns={"question_representation": "embed_text"})
#
# vectors = []
# for batch in batched(qid_text_df["embed_text"].tolist(), batch_size):
#     resp = client.embeddings.create(model=embed_model, input=batch)
#     vectors.extend([row.embedding for row in resp.data])
#
# qid_text_df["embedding"] = [np.asarray(v, dtype=float).tolist() for v in vectors]
# out_path = REPRO_ROOT / "data/worldvalue/simulator_selector/wvs_qid_embeddings_openai_text-embedding-3-small.pkl"
# with open(out_path, "wb") as f:
#     pickle.dump(qid_text_df[["qid", "question_text", "category", "embed_text", "embedding"]], f)
# print(f"saved: {out_path}")


In [ ]:
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import Markdown, display
from sklearn.metrics import mean_absolute_error, mean_squared_error, pairwise_distances
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.feature_extraction.text import TfidfVectorizer

from wvs_notebook_helpers import (
    ensure_worldvalue_inputs,
    filter_mapping_to_questions,
    find_repo_root,
    install_numpy_pickle_compat,
)
from simfidelity_utils import compute_pseudo_delta, empirical_quantile_curve

install_numpy_pickle_compat()
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

NOTEBOOK_DIR = Path.cwd().resolve()
REPRO_ROOT = find_repo_root(NOTEBOOK_DIR)
ensure_worldvalue_inputs(REPRO_ROOT)
OUTPUT_DIR = NOTEBOOK_DIR / "output_embedding_benchmark"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = NOTEBOOK_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_PATHS = {
    "source_notebook": NOTEBOOK_DIR / "WV_quantile_construction.ipynb",
    "retained_questions": REPRO_ROOT / "data/worldvalue/retained_questions_235.json",
    "full_human": REPRO_ROOT / "data/worldvalue/full_population_response_clean.pkl",
    "full_summary": REPRO_ROOT / "data/worldvalue/full_population_response_summary.pkl",
    "actual_human": REPRO_ROOT / "data/worldvalue/population_response_clean.pkl",
    "question_metadata": REPRO_ROOT / "data/worldvaluesbench/dataset_construction/question_metadata.json",
    "codebook": REPRO_ROOT / "data/worldvaluesbench/dataset_construction/codebook.json",
    "value_questions": REPRO_ROOT / "data/worldvaluesbench/dataset_construction/probe_set_construction/value_questions.json",
    "cached_openai_embeddings": REPRO_ROOT / "data/worldvalue/simulator_selector/wvs_qid_embeddings_openai_text-embedding-3-small.pkl",
}

display(
    pd.DataFrame(
        {
            "name": list(SOURCE_PATHS.keys()),
            "path": [str(path.relative_to(REPRO_ROOT)) for path in SOURCE_PATHS.values()],
            "exists": [path.exists() for path in SOURCE_PATHS.values()],
        }
    )
)
print("Note: this notebook prefers the repo's cached OpenAI embedding artifact when it fully covers the retained qids.")


### Load the same WorldValue objects as the calibrated notebook


In [ ]:
with open(SOURCE_PATHS["retained_questions"], "r", encoding="utf-8") as f:
    retained_questions = [str(q) for q in json.load(f)]
retained_qid_set = set(retained_questions)

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

gpt4o = filter_mapping_to_questions(
    load_pickle(REPRO_ROOT / "data/worldvalue/synthetic answers/clean/synthetic_answers_clean_gpt4o.pkl"),
    retained_questions,
)
gpt5m = filter_mapping_to_questions(
    load_pickle(REPRO_ROOT / "data/worldvalue/synthetic answers/clean/synthetic_answers_clean_gpt-5-mini.pkl"),
    retained_questions,
)
llama = filter_mapping_to_questions(
    load_pickle(REPRO_ROOT / "data/worldvalue/synthetic answers/clean/synthetic_answers_clean_llama-3.3.pkl"),
    retained_questions,
)
qwen = filter_mapping_to_questions(
    load_pickle(REPRO_ROOT / "data/worldvalue/synthetic answers/clean/synthetic_answers_clean_Qwen3-235B.pkl"),
    retained_questions,
)
uniform_benchmark = filter_mapping_to_questions(
    load_pickle(REPRO_ROOT / "data/worldvalue/synthetic answers/clean/uniform_benchmark.pkl"),
    retained_questions,
)

actual = filter_mapping_to_questions(load_pickle(SOURCE_PATHS["actual_human"]), retained_questions)
full = filter_mapping_to_questions(load_pickle(SOURCE_PATHS["full_human"]), retained_questions)
full_summary = load_pickle(SOURCE_PATHS["full_summary"])

with open(SOURCE_PATHS["question_metadata"], "r", encoding="utf-8") as f:
    question_metadata = json.load(f)
with open(SOURCE_PATHS["codebook"], "r", encoding="utf-8") as f:
    codebook = json.load(f)
if SOURCE_PATHS["value_questions"].exists():
    with open(SOURCE_PATHS["value_questions"], "r", encoding="utf-8") as f:
        value_questions = json.load(f)
else:
    warnings.warn("value_questions.json is unavailable in the reproduction tree; proceeding without it.")
    value_questions = {}

simulators = {
    "GPT4o": gpt4o,
    "Llama": llama,
    "GPT5mini": gpt5m,
    "Qwen3": qwen,
    "Uniform": uniform_benchmark,
}
SIM_ORDER = ["GPT4o", "GPT5mini", "Llama", "Qwen3", "Uniform"]
SIM_COLOR_MAP = {
    "GPT4o": "C0",
    "Llama": "C1",
    "GPT5mini": "C2",
    "Uniform": "C3",
    "Qwen3": "C4",
}

full_summary_df = full_summary.copy()
if "qid" not in full_summary_df.columns:
    full_summary_df = full_summary_df.reset_index()
if "index" in full_summary_df.columns and "qid" not in full_summary_df.columns:
    full_summary_df = full_summary_df.rename(columns={"index": "qid"})
full_summary_df["qid"] = full_summary_df["qid"].astype(str).str.strip()
full_summary_df = full_summary_df[full_summary_df["qid"].isin(retained_qid_set)].copy()
full_summary_df = full_summary_df.set_index("qid").reindex(retained_questions).reset_index()

display(
    pd.DataFrame(
        {
            "object": ["retained_questions", "full", "actual", *SIM_ORDER, "full_summary_df"],
            "n_items": [len(retained_questions), len(full), len(actual), *(len(simulators[s]) for s in SIM_ORDER), len(full_summary_df)],
        }
    )
)


### Select the strongest human-side real-data target

The predictive benchmark is intended to answer the reviewer-style question: how much of the human-side target can be fit directly from real data?

Selection hierarchy

- First choice: raw full-data human responses for the retained qids
- Second choice: full-data question-level aggregates
- Last resort: smaller-sample human targets

This keeps the predictive benchmark focused on estimating `p`, while the simulator-side quantity `qhat` remains fixed exactly as in the calibrated discrepancy pipeline.


In [ ]:
def summarize_response_mapping(resp_dict, source_name):
    rows = []
    for qid in retained_questions:
        raw = resp_dict.get(qid, [])
        arr = np.asarray(getattr(raw, "values", raw), dtype=float)
        arr = arr[np.isfinite(arr)]
        rows.append(
            {
                "qid": qid,
                "source_name": source_name,
                "n_valid": int(arr.size),
                "mean": float(arr.mean()) if arr.size else np.nan,
                "std": float(arr.std(ddof=1)) if arr.size >= 2 else np.nan,
            }
        )
    return pd.DataFrame(rows)

full_raw_target_df = summarize_response_mapping(full, "full_raw_responses")
actual_target_df = summarize_response_mapping(actual, "actual_smaller_sample")
full_summary_target_df = full_summary_df.rename(columns={"source_name": "unused"}).copy()
full_summary_target_df["source_name"] = "full_summary_aggregate"
if "n_valid" not in full_summary_target_df.columns:
    full_summary_target_df["n_valid"] = np.nan
full_summary_target_df = full_summary_target_df[["qid", "source_name", "n_valid", "mean", "std"]]

target_candidates = []
for candidate_name, candidate_df in [
    ("full_raw_responses", full_raw_target_df),
    ("full_summary_aggregate", full_summary_target_df),
    ("actual_smaller_sample", actual_target_df),
]:
    target_candidates.append(
        {
            "candidate_name": candidate_name,
            "n_qids_with_target": int(candidate_df["mean"].notna().sum()),
            "min_n_valid": float(np.nanmin(candidate_df["n_valid"])) if candidate_df["n_valid"].notna().any() else np.nan,
            "max_n_valid": float(np.nanmax(candidate_df["n_valid"])) if candidate_df["n_valid"].notna().any() else np.nan,
        }
    )

target_selection_df = pd.DataFrame(target_candidates)
if full_raw_target_df["mean"].notna().sum() == len(retained_questions):
    selected_target_df = full_raw_target_df.copy()
    selected_target_source = "full_raw_responses"
elif full_summary_target_df["mean"].notna().sum() == len(retained_questions):
    selected_target_df = full_summary_target_df.copy()
    selected_target_source = "full_summary_aggregate"
else:
    selected_target_df = actual_target_df.copy()
    selected_target_source = "actual_smaller_sample"

selected_target_df = selected_target_df.rename(
    columns={
        "mean": "human_target",
        "std": "human_target_std",
        "n_valid": "human_target_n",
    }
)
selected_target_df["human_target_source"] = selected_target_source

full_target_consistency_df = full_raw_target_df.merge(
    full_summary_target_df.rename(
        columns={
            "mean": "summary_mean",
            "std": "summary_std",
            "n_valid": "summary_n_valid",
        }
    ),
    on="qid",
    how="inner",
)
full_target_consistency_df["abs_mean_diff"] = (
    full_target_consistency_df["mean"] - full_target_consistency_df["summary_mean"]
).abs()
full_target_consistency_summary = pd.DataFrame(
    [
        {
            "selected_target_source": selected_target_source,
            "max_abs_mean_diff_raw_vs_summary": float(full_target_consistency_df["abs_mean_diff"].max()),
            "mean_abs_mean_diff_raw_vs_summary": float(full_target_consistency_df["abs_mean_diff"].mean()),
        }
    ]
)

target_selection_df.to_csv(OUTPUT_DIR / "human_target_source_summary.csv", index=False)
full_target_consistency_summary.to_csv(OUTPUT_DIR / "human_target_consistency_summary.csv", index=False)

display(target_selection_df)
display(full_target_consistency_summary)
print("Selected predictive target source:", selected_target_source)


### Build the question-level feature dataframe

Predictors use question semantics and metadata only for `plugin_X`, and question semantics plus the simulator estimate `qhat` for `plugin_XQ`.


In [ ]:
def _safe_float(value):
    if value in ("", None):
        return np.nan
    try:
        return float(value)
    except Exception:
        return np.nan

def _clean_choices(choices):
    if not isinstance(choices, dict):
        return {}

    def sort_key(item):
        key = str(item[0])
        try:
            return (0, float(key))
        except Exception:
            return (1, key)

    out = {}
    for raw_key, raw_val in sorted(choices.items(), key=sort_key):
        key = str(raw_key)
        if key.startswith("-"):
            continue
        val = str(raw_val).strip()
        if val:
            out[key] = val
    return out

question_rows = []
for qid in retained_questions:
    meta = question_metadata.get(qid, {})
    cb = codebook.get(qid, {})
    choice_dict = _clean_choices(cb.get("choices"))
    question_rows.append(
        {
            "qid": qid,
            "question_text": str(meta.get("question") or cb.get("question") or qid).strip(),
            "short_label": str(cb.get("question") or "").strip(),
            "question_instruction": str(cb.get("question_instruction") or "").strip(),
            "answer_option_text": " | ".join([f"{k} = {v}" for k, v in choice_dict.items()]),
            "category": str(meta.get("category") or "Unknown").strip(),
            "use_case": str(meta.get("use_case") or "").strip(),
            "answer_data_type": str(meta.get("answer_data_type") or cb.get("type") or "").strip(),
            "answer_scale_min": _safe_float(meta.get("answer_scale_min")),
            "answer_scale_max": _safe_float(meta.get("answer_scale_max")),
            "n_answer_choices": len(choice_dict),
            "in_value_questions_json": qid in set(value_questions.keys()) if isinstance(value_questions, dict) else qid in set(value_questions),
        }
    )

question_df = pd.DataFrame(question_rows).merge(
    selected_target_df[["qid", "human_target", "human_target_std", "human_target_n", "human_target_source"]],
    on="qid",
    how="left",
)
question_df["scale_span"] = question_df["answer_scale_max"] - question_df["answer_scale_min"]
question_df["question_representation"] = (
    "Category: " + question_df["category"].fillna("Unknown") + "\n"
    + "Question: " + question_df["question_text"].fillna(question_df["qid"]) + "\n"
    + np.where(
        question_df["question_instruction"].fillna("").ne(""),
        "Instruction: " + question_df["question_instruction"].fillna("") + "\n",
        "",
    )
    + np.where(
        question_df["answer_option_text"].fillna("").ne(""),
        "Answer options: " + question_df["answer_option_text"].fillna(""),
        "Answer options: unavailable",
    )
)

missing_text_mask = question_df["question_text"].fillna("").str.strip().eq("")
missing_options_mask = question_df["answer_option_text"].fillna("").str.strip().eq("")
if missing_text_mask.any():
    warnings.warn(f"{int(missing_text_mask.sum())} retained qids are missing question text and will fall back to qid-only text.")
if missing_options_mask.any():
    warnings.warn(f"{int(missing_options_mask.sum())} retained qids are missing answer-option text.")

display(question_df.head(8))


### Build question representations and define the target

Methodological notes

- We predict `p`, the human-side question target, not `q`.
- The selected target comes from the strongest retained-qid real-data source available in the repo, which is more faithful to a “fit on real data” benchmark than using the smaller-sample human object when full data exists.
- `f(X)` asks how much of the human target is predictable from question semantics and metadata alone.
- `f(X, qhat)` is a stronger benchmark because it gives the predictor access to the simulator-side estimate already central to the discrepancy.
- Predictive curves are plug-in references, not coverage-calibrated upper envelopes.
- A predictive curve can lie above a calibrated curve without implying a bug, because the predictive task still has to learn across only 235 questions.


In [ ]:
def build_structured_metadata_frame(df: pd.DataFrame) -> pd.DataFrame:
    base = df[
        [
            "n_answer_choices",
            "answer_scale_min",
            "answer_scale_max",
            "scale_span",
        ]
    ].copy().fillna(0.0)
    base["is_ordinal"] = df["answer_data_type"].astype(str).str.lower().eq("ordinal").astype(int)
    base["has_instruction"] = df["question_instruction"].astype(str).str.strip().ne("").astype(int)
    base["has_answer_options"] = df["answer_option_text"].astype(str).str.strip().ne("").astype(int)

    category_onehot = pd.get_dummies(df["category"].fillna("Unknown").astype(str), prefix="category")
    use_case_onehot = pd.get_dummies(df["use_case"].fillna("Unknown").astype(str), prefix="use_case")
    out = pd.concat([base.astype(float), category_onehot, use_case_onehot], axis=1)
    out.index = df["qid"].tolist()
    return out

def try_cached_openai_embeddings(qid_df: pd.DataFrame, embedding_path: Path):
    runtime_log = []
    if not embedding_path.exists():
        runtime_log.append("Cached OpenAI embedding file is missing.")
        return None, None, runtime_log
    try:
        with open(embedding_path, "rb") as f:
            cached_obj = pickle.load(f)
    except Exception as exc:
        runtime_log.append(f"Could not load cached OpenAI embeddings: {type(exc).__name__}: {exc}")
        return None, None, runtime_log

    if not isinstance(cached_obj, pd.DataFrame):
        runtime_log.append(f"Cached embedding object has unexpected type: {type(cached_obj).__name__}")
        return None, None, runtime_log

    required_cols = {"qid", "embedding"}
    if not required_cols.issubset(set(cached_obj.columns)):
        runtime_log.append(f"Cached embedding DataFrame missing required columns: {sorted(required_cols - set(cached_obj.columns))}")
        return None, None, runtime_log

    cached_df = cached_obj.copy()
    cached_df["qid"] = cached_df["qid"].astype(str).str.strip()
    cached_df = cached_df.drop_duplicates(subset=["qid"], keep="first")

    merged = qid_df[["qid", "question_representation"]].merge(
        cached_df[["qid", "embedding", "embed_text"]] if "embed_text" in cached_df.columns else cached_df[["qid", "embedding"]],
        on="qid",
        how="left",
    )
    missing_qids = merged.loc[merged["embedding"].isna(), "qid"].tolist()
    if missing_qids:
        runtime_log.append(f"Cached OpenAI embeddings are missing {len(missing_qids)} retained qids.")
        return None, None, runtime_log

    arrays = []
    dims = set()
    for raw_emb in merged["embedding"]:
        arr = np.asarray(raw_emb, dtype=float).ravel()
        arrays.append(arr)
        dims.add(arr.size)
    if len(dims) != 1:
        runtime_log.append(f"Cached OpenAI embeddings have inconsistent dimensions: {sorted(dims)}")
        return None, None, runtime_log

    dense = np.vstack(arrays).astype(np.float64)
    runtime_log.append(
        f"Using cached OpenAI embeddings from {embedding_path.name} with dimension {dense.shape[1]}."
    )
    if "embed_text" in merged.columns:
        embed_text_match_rate = float(
            np.mean(
                merged["embed_text"].fillna("").astype(str).str.strip()
                == merged["question_representation"].fillna("").astype(str).str.strip()
            )
        )
        runtime_log.append(f"Exact cached-embed-text match rate vs current question representation: {embed_text_match_rate:.3f}")
    return dense, "cached_openai_embeddings:text-embedding-3-small", runtime_log

def try_local_sentence_transformer_embeddings(texts):
    runtime_log = []
    try:
        from sentence_transformers import SentenceTransformer
    except Exception as exc:
        runtime_log.append(f"sentence_transformers unavailable: {type(exc).__name__}: {exc}")
        return None, None, runtime_log

    for model_name in ["BAAI/bge-m3", "sentence-transformers/all-MiniLM-L6-v2"]:
        try:
            model = SentenceTransformer(model_name, device="cpu", local_files_only=True)
            emb = model.encode(texts, batch_size=32, show_progress_bar=False, normalize_embeddings=False)
            runtime_log.append(f"Using local sentence-transformer model: {model_name}")
            return np.asarray(emb, dtype=float), f"local_sentence_transformer:{model_name}", runtime_log
        except Exception as exc:
            runtime_log.append(f"{model_name}: {type(exc).__name__}: {exc}")
    return None, None, runtime_log

texts = question_df["question_representation"].tolist()
dense_text_features, embedding_source, embedding_runtime_log = try_cached_openai_embeddings(
    question_df,
    SOURCE_PATHS["cached_openai_embeddings"],
)
if dense_text_features is None:
    dense_text_features, embedding_source, local_runtime_log = try_local_sentence_transformer_embeddings(texts)
    embedding_runtime_log.extend(local_runtime_log)
if dense_text_features is None:
    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_features=4000, sublinear_tf=True)
    dense_text_features = tfidf.fit_transform(texts).astype(np.float64).toarray()
    embedding_source = "tfidf_fallback"
    embedding_runtime_log.append("Falling back to TF-IDF text features.")

metadata_df = build_structured_metadata_frame(question_df)
metadata_matrix = metadata_df.reindex(question_df["qid"]).to_numpy(dtype=float)
X_question = np.hstack([dense_text_features, metadata_matrix]).astype(np.float64)

y_question = question_df["human_target"].to_numpy(dtype=float)
if np.isnan(y_question).any():
    raise ValueError("The human target must be finite for every retained qid.")

feature_source_df = question_df[
    [
        "qid",
        "category",
        "question_text",
        "answer_option_text",
        "question_instruction",
        "human_target",
        "human_target_std",
        "human_target_n",
        "human_target_source",
    ]
].copy()
feature_source_df["embedding_source"] = embedding_source
feature_source_df["text_feature_dim"] = dense_text_features.shape[1]
feature_source_df["metadata_feature_dim"] = metadata_matrix.shape[1]
feature_source_df["combined_feature_dim"] = X_question.shape[1]
feature_source_df["question_representation"] = question_df["question_representation"]
feature_source_df.to_csv(OUTPUT_DIR / "question_feature_dataframe.csv", index=False)

print("Embedding source:", embedding_source)
print("Text feature dim:", dense_text_features.shape[1])
print("Metadata feature dim:", metadata_matrix.shape[1])
print("Combined feature dim:", X_question.shape[1])
print("Embedding runtime log")
for msg in embedding_runtime_log:
    print(f"- {msg}")

display(feature_source_df.head(6))


## SECTION 2 — Modeling

This section keeps the unchanged calibrated pipeline and the new predictive models together, so it is easy to see that both benchmark families share the same retained questions and the same simulator-side `qhat`.


In [ ]:
CI_FAMILY = "bounded"
LOSS_KIND = "sq"
GAMMA = 0.50
K_MODEL = 200
N_HUMAN = 500
SEED = 0
TAU_GRID = np.linspace(0.0, 1.0, 201)
ALPHA_RAW_GRID = np.linspace(0.0, 1.0, 201)
TAIL_ALPHA0 = 0.90
CI_KWARGS = {"bounds": (-1.0, 1.0), "method": "hoeffding"}

def _coerce_1d_float_array(x):
    if hasattr(x, "values"):
        x = x.values
    return np.asarray(x, dtype=float).ravel()

def _subsample_human(yh, n_target=None, rng=None, replace=False):
    if rng is None:
        rng = np.random.default_rng()
    y = _coerce_1d_float_array(yh)
    y = y[np.isfinite(y)]
    n_avail = int(y.size)
    if n_avail == 0:
        return np.asarray([], dtype=float), 0
    if (n_target is None) or (int(n_target) <= 0):
        return y, n_avail
    n_use = int(min(int(n_target), n_avail))
    if (not replace) and (n_use == n_avail):
        return y, n_avail
    idx = rng.choice(n_avail, size=n_use, replace=bool(replace))
    return y[idx], n_use

def gamma_schedule_power_1_3(n_eff: int, eps: float = 1e-6) -> float:
    n_eff = int(n_eff)
    if n_eff <= 0:
        return np.nan
    return float(np.clip(1.0 - (n_eff ** (-1.0 / 3.0)), eps, 1.0 - eps))

def compute_deltas_multi_with_details(
    actual_dict,
    simulators_dict,
    k=500,
    n_target=None,
    ci_family="bounded",
    loss_kind="sq",
    ci_kwargs=None,
    seed=0,
    human_replace=False,
):
    ci_kwargs = ci_kwargs or {}
    common = set(actual_dict.keys())
    for sim in simulators_dict.values():
        common &= set(sim.keys())
    common = sorted(common)

    rng_master = np.random.default_rng(seed)
    rows = []
    qhat_rows = []
    deltas_dict = {}
    human_sub = {}
    n_eff_by_qid = {}
    gamma_by_qid = {}
    seed_by_qid = {}

    for qid in common:
        local_seed = int(rng_master.integers(0, 2**32 - 1))
        rng_local = np.random.default_rng(local_seed)
        yh_sub, n_eff = _subsample_human(actual_dict[qid], n_target=n_target, rng=rng_local, replace=human_replace)
        if n_eff <= 0:
            continue
        gamma_j = gamma_schedule_power_1_3(n_eff)
        human_sub[qid] = yh_sub
        n_eff_by_qid[qid] = int(n_eff)
        gamma_by_qid[qid] = float(gamma_j)
        seed_by_qid[qid] = int(local_seed)

    used_qids = sorted(human_sub.keys())
    for sim_idx, (sim_name, sim_map) in enumerate(simulators_dict.items()):
        sim_dict = {}
        for qid in used_qids:
            ym = sim_map[qid]
            if hasattr(ym, "values"):
                ym = ym.values
            rng_local = np.random.default_rng(seed_by_qid[qid] + 1009 * (sim_idx + 1))
            delta_val, info = compute_pseudo_delta(
                y_human=human_sub[qid],
                y_model=ym,
                k=int(k),
                gamma=float(gamma_by_qid[qid]),
                ci_family=ci_family,
                loss_kind=loss_kind,
                ci_kwargs=ci_kwargs,
                rng=rng_local,
            )
            qhat_rows.append(
                {
                    "qid": qid,
                    "sim": sim_name,
                    "qhat": float(info.get("qhat", np.nan)),
                    "k_used": float(info.get("k_used", np.nan)),
                    "n_eff": int(n_eff_by_qid[qid]),
                    "gamma_j": float(gamma_by_qid[qid]),
                }
            )
            if np.isfinite(delta_val):
                sim_dict[qid] = float(delta_val)
                rows.append(
                    {
                        "qid": qid,
                        "sim": sim_name,
                        "delta": float(delta_val),
                        "n_eff": int(n_eff_by_qid[qid]),
                        "k": int(k),
                        "gamma_j": float(gamma_by_qid[qid]),
                    }
                )
        deltas_dict[sim_name] = sim_dict

    delta_df = pd.DataFrame(rows)
    qhat_df = pd.DataFrame(qhat_rows)
    gamma_summary = {
        "bar_gamma_overall": float(delta_df["gamma_j"].mean()),
        "bar_gamma_by_sim": {str(k): float(v) for k, v in delta_df.groupby("sim", sort=True)["gamma_j"].mean().items()},
    }
    return deltas_dict, used_qids, delta_df, qhat_df, gamma_summary

calibrated_dict, common_qids, calibrated_delta_df, simulator_qhat_df, gamma_summary = compute_deltas_multi_with_details(
    actual_dict=full,
    simulators_dict=simulators,
    k=K_MODEL,
    n_target=N_HUMAN,
    ci_family=CI_FAMILY,
    loss_kind=LOSS_KIND,
    ci_kwargs=CI_KWARGS,
    seed=SEED,
    human_replace=False,
)

calibrated_delta_df.to_csv(OUTPUT_DIR / "calibrated_delta_dataframe.csv", index=False)
simulator_qhat_df.to_csv(OUTPUT_DIR / "simulator_qhat_dataframe.csv", index=False)

expected_qids = set(question_df["qid"])
calibrated_qids = set(common_qids)
qhat_qids = set(simulator_qhat_df["qid"])
if expected_qids != calibrated_qids:
    raise AssertionError(
        f"Question feature qids and calibrated qids differ. "
        f"Only-in-features={sorted(expected_qids - calibrated_qids)[:10]}, "
        f"only-in-calibrated={sorted(calibrated_qids - expected_qids)[:10]}"
    )
if expected_qids != qhat_qids:
    raise AssertionError(
        f"Question feature qids and qhat qids differ. "
        f"Only-in-features={sorted(expected_qids - qhat_qids)[:10]}, "
        f"only-in-qhat={sorted(qhat_qids - expected_qids)[:10]}"
    )

display(calibrated_delta_df.head())
print(gamma_summary)


### Cross-fitted predictive baselines

Two predictors are run:

- `plugin_X`: one shared question-only model `f(X)`
- `plugin_XQ`: one simulator-specific model per simulator `f_sim(X, qhat)`

Both predict the human target `p`.


In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def l2_normalize(X: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms = np.where(norms <= 1e-12, 1.0, norms)
    return X / norms

def median_heuristic_gamma(X: np.ndarray) -> float:
    if X.shape[0] <= 1:
        return 1.0
    D = pairwise_distances(X, metric="euclidean")
    tri = D[np.triu_indices_from(D, k=1)]
    tri = tri[np.isfinite(tri) & (tri > 0)]
    if tri.size == 0:
        return 1.0
    med = float(np.median(tri))
    return 1.0 / max(med ** 2, 1e-6)

def candidate_grid(X_train: np.ndarray):
    gamma0 = median_heuristic_gamma(X_train)
    grid = []
    for k in [3, 5, 10, 20, 40]:
        if k < X_train.shape[0]:
            grid.append({"model_type": "knn_cosine", "n_neighbors": int(k)})
    for gamma_mult in [0.25, 0.5, 1.0, 2.0]:
        for alpha in [1e-3, 1e-2]:
            grid.append({"model_type": "kernel_ridge_rbf", "gamma": float(gamma0 * gamma_mult), "alpha": float(alpha)})
    return grid

def fit_predict(candidate: dict, X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray) -> np.ndarray:
    if candidate["model_type"] == "knn_cosine":
        model = KNeighborsRegressor(
            n_neighbors=int(candidate["n_neighbors"]),
            metric="cosine",
            weights="distance",
        )
        model.fit(l2_normalize(X_train), y_train)
        return model.predict(l2_normalize(X_test))
    if candidate["model_type"] == "kernel_ridge_rbf":
        model = KernelRidge(kernel="rbf", gamma=float(candidate["gamma"]), alpha=float(candidate["alpha"]))
        model.fit(X_train, y_train)
        return model.predict(X_test)
    raise ValueError(candidate)

def select_model(X_train: np.ndarray, y_train: np.ndarray, seed: int) -> dict:
    inner_cv = KFold(n_splits=4, shuffle=True, random_state=seed)
    rows = []
    for cand in candidate_grid(X_train):
        fold_rmse = []
        fold_mae = []
        for inner_train_idx, inner_val_idx in inner_cv.split(X_train):
            pred = fit_predict(cand, X_train[inner_train_idx], y_train[inner_train_idx], X_train[inner_val_idx])
            fold_rmse.append(rmse(y_train[inner_val_idx], pred))
            fold_mae.append(float(mean_absolute_error(y_train[inner_val_idx], pred)))
        rows.append({**cand, "cv_rmse": float(np.mean(fold_rmse)), "cv_mae": float(np.mean(fold_mae))})
    score_df = pd.DataFrame(rows).sort_values(["cv_rmse", "cv_mae"]).reset_index(drop=True)
    return score_df.iloc[0].to_dict()

OUTER_FOLDS = list(KFold(n_splits=5, shuffle=True, random_state=0).split(question_df["qid"].tolist()))

def assert_no_leakage(qids, splits):
    for fold_id, (train_idx, test_idx) in enumerate(splits, start=1):
        train_qids = {qids[i] for i in train_idx}
        test_qids = {qids[i] for i in test_idx}
        overlap = train_qids & test_qids
        if overlap:
            raise AssertionError(f"Fold {fold_id} leakage detected: {sorted(list(overlap))[:5]}")

assert_no_leakage(question_df["qid"].tolist(), OUTER_FOLDS)

def run_crossfit(X: np.ndarray, y: np.ndarray, qids: list[str], splits, seed_offset: int = 0):
    oof = np.full(shape=y.shape, fill_value=np.nan, dtype=float)
    pred_rows = []
    fold_rows = []
    for fold, (train_idx, test_idx) in enumerate(splits, start=1):
        best = select_model(X[train_idx], y[train_idx], seed_offset + fold)
        pred = fit_predict(best, X[train_idx], y[train_idx], X[test_idx])
        oof[test_idx] = pred
        fold_rows.append(
            {
                "fold": fold,
                "selected_model_type": best["model_type"],
                "selected_params": json.dumps({k: v for k, v in best.items() if k not in {"cv_rmse", "cv_mae"}}, sort_keys=True),
                "inner_cv_rmse": float(best["cv_rmse"]),
                "inner_cv_mae": float(best["cv_mae"]),
                "n_train": int(len(train_idx)),
                "n_test": int(len(test_idx)),
            }
        )
        for idx, pred_val in zip(test_idx, pred):
            pred_rows.append(
                {
                    "qid": qids[idx],
                    "fold": fold,
                    "human_target_true": float(y[idx]),
                    "human_target_pred_oof": float(pred_val),
                }
            )
    if np.isnan(oof).any():
        raise RuntimeError("Cross-fitting did not produce predictions for every qid.")
    return oof, pd.DataFrame(pred_rows), pd.DataFrame(fold_rows)

# Question-only predictor
oof_X, pred_X_df, folds_X_df = run_crossfit(
    X=X_question,
    y=y_question,
    qids=question_df["qid"].tolist(),
    splits=OUTER_FOLDS,
    seed_offset=100,
)
pred_X_df["benchmark_type"] = "plugin_X"
pred_X_df["global_mean_baseline"] = float(np.mean(y_question))
pred_X_df["baseline_abs_error"] = (pred_X_df["human_target_true"] - pred_X_df["global_mean_baseline"]).abs()
pred_X_df["residual"] = pred_X_df["human_target_true"] - pred_X_df["human_target_pred_oof"]
pred_X_df["abs_error"] = pred_X_df["residual"].abs()
pred_X_df["sq_error"] = pred_X_df["residual"] ** 2
pred_X_df = pred_X_df.merge(question_df[["qid", "category", "question_text", "human_target_source"]], on="qid", how="left")
pred_X_df.to_csv(OUTPUT_DIR / "oof_prediction_question_only.csv", index=False)

predictor_summary_rows = [
    {
        "benchmark_type": "plugin_X",
        "sim": "shared",
        "human_target_source": selected_target_source,
        "embedding_source": embedding_source,
        "overall_rmse": rmse(pred_X_df["human_target_true"], pred_X_df["human_target_pred_oof"]),
        "overall_mae": float(mean_absolute_error(pred_X_df["human_target_true"], pred_X_df["human_target_pred_oof"])),
        "baseline_rmse_global_mean": float(np.sqrt(np.mean((pred_X_df["human_target_true"] - pred_X_df["global_mean_baseline"]) ** 2))),
        "baseline_mae_global_mean": float(np.mean(pred_X_df["baseline_abs_error"])),
        "selected_model_counts": json.dumps(folds_X_df["selected_model_type"].value_counts().to_dict(), sort_keys=True),
    }
]

display(pred_X_df.head())
display(pd.DataFrame(predictor_summary_rows))


In [ ]:
# Simulator-assisted predictors: one model per simulator using [X, qhat_sim]
pred_XQ_rows = []
folds_XQ_rows = []

qhat_aligned_df = (
    simulator_qhat_df[["qid", "sim", "qhat"]]
    .copy()
    .dropna(subset=["qhat"])
)

for sim_name in SIM_ORDER:
    sub = qhat_aligned_df[qhat_aligned_df["sim"] == sim_name].copy()
    sub = sub.set_index("qid").reindex(question_df["qid"]).reset_index()
    if sub["qhat"].isna().any():
        missing_qids = sub.loc[sub["qhat"].isna(), "qid"].tolist()
        raise ValueError(f"Missing qhat for {sim_name}: {missing_qids[:10]}")

    X_sim = np.hstack([X_question, sub[["qhat"]].to_numpy(dtype=float)])
    oof_XQ, pred_sim_df, folds_sim_df = run_crossfit(
        X=X_sim,
        y=y_question,
        qids=question_df["qid"].tolist(),
        splits=OUTER_FOLDS,
        seed_offset=1000 + SIM_ORDER.index(sim_name) * 100,
    )
    pred_sim_df["benchmark_type"] = "plugin_XQ"
    pred_sim_df["sim"] = sim_name
    pred_sim_df["qhat"] = sub.set_index("qid").loc[pred_sim_df["qid"], "qhat"].to_numpy()
    pred_sim_df["global_mean_baseline"] = float(np.mean(y_question))
    pred_sim_df["baseline_abs_error"] = (pred_sim_df["human_target_true"] - pred_sim_df["global_mean_baseline"]).abs()
    pred_sim_df["residual"] = pred_sim_df["human_target_true"] - pred_sim_df["human_target_pred_oof"]
    pred_sim_df["abs_error"] = pred_sim_df["residual"].abs()
    pred_sim_df["sq_error"] = pred_sim_df["residual"] ** 2
    pred_XQ_rows.append(pred_sim_df)

    folds_sim_df["sim"] = sim_name
    folds_XQ_rows.append(folds_sim_df)

    predictor_summary_rows.append(
        {
            "benchmark_type": "plugin_XQ",
            "sim": sim_name,
            "human_target_source": selected_target_source,
            "embedding_source": embedding_source,
            "overall_rmse": rmse(pred_sim_df["human_target_true"], pred_sim_df["human_target_pred_oof"]),
            "overall_mae": float(mean_absolute_error(pred_sim_df["human_target_true"], pred_sim_df["human_target_pred_oof"])),
            "baseline_rmse_global_mean": float(np.sqrt(np.mean((pred_sim_df["human_target_true"] - pred_sim_df["global_mean_baseline"]) ** 2))),
            "baseline_mae_global_mean": float(np.mean(pred_sim_df["baseline_abs_error"])),
            "selected_model_counts": json.dumps(folds_sim_df["selected_model_type"].value_counts().to_dict(), sort_keys=True),
        }
    )

pred_XQ_df = pd.concat(pred_XQ_rows, ignore_index=True)
folds_XQ_df = pd.concat(folds_XQ_rows, ignore_index=True)
pred_XQ_df = pred_XQ_df.merge(question_df[["qid", "category", "question_text", "human_target_source"]], on="qid", how="left")
pred_XQ_df.to_csv(OUTPUT_DIR / "oof_prediction_simulator_assisted.csv", index=False)

predictor_summary_df = pd.DataFrame(predictor_summary_rows)
predictor_summary_df.to_csv(OUTPUT_DIR / "predictor_summary.csv", index=False)
folds_X_df.assign(sim="shared").to_csv(OUTPUT_DIR / "crossfit_fold_selection_question_only.csv", index=False)
folds_XQ_df.to_csv(OUTPUT_DIR / "crossfit_fold_selection_simulator_assisted.csv", index=False)

display(pred_XQ_df.head())
display(predictor_summary_df)


## SECTION 3 — Curves, Figures, and Diagnostics

The predictive benchmarks remain of the form `loss(p_tilde, qhat)`.

Indexing distinction

- `calibrated`: plotted against `tau`, but evaluated internally at the theorem-shifted quantile index `alpha(tau) = (1 - bar_gamma) + bar_gamma * tau`
- `plugin_X`, `plugin_XQ`: plotted against the raw empirical quantile index, so their displayed `tau` is also their evaluation quantile


In [ ]:
plugin_X_base = simulator_qhat_df[["qid", "sim", "qhat"]].merge(
    pred_X_df[["qid", "human_target_true", "human_target_pred_oof"]],
    on="qid",
    how="left",
)
plugin_X_base["curve_type"] = "plugin_X"
plugin_X_base["plugin_delta"] = (plugin_X_base["human_target_pred_oof"] - plugin_X_base["qhat"]) ** 2
plugin_X_base["oracle_sq_loss"] = (plugin_X_base["human_target_true"] - plugin_X_base["qhat"]) ** 2

plugin_XQ_base = pred_XQ_df[["qid", "sim", "qhat", "human_target_true", "human_target_pred_oof"]].copy()
plugin_XQ_base["curve_type"] = "plugin_XQ"
plugin_XQ_base["plugin_delta"] = (plugin_XQ_base["human_target_pred_oof"] - plugin_XQ_base["qhat"]) ** 2
plugin_XQ_base["oracle_sq_loss"] = (plugin_XQ_base["human_target_true"] - plugin_XQ_base["qhat"]) ** 2

oracle_emp_base = (
    simulator_qhat_df[["qid", "sim", "qhat"]]
    .merge(selected_target_df[["qid", "human_target"]], on="qid", how="left")
    .rename(columns={"human_target": "p_full_target"})
)
oracle_emp_base["curve_type"] = "oracle_empirical"
oracle_emp_base["oracle_emp_delta"] = (oracle_emp_base["p_full_target"] - oracle_emp_base["qhat"]) ** 2

plugin_delta_df = pd.concat([plugin_X_base, plugin_XQ_base], ignore_index=True)
plugin_delta_df = plugin_delta_df.merge(
    question_df[["qid", "category", "question_text", "answer_option_text"]],
    on="qid",
    how="left",
)
plugin_delta_df.to_csv(OUTPUT_DIR / "plugin_delta_dataframe.csv", index=False)
oracle_emp_base.to_csv(OUTPUT_DIR / "oracle_empirical_delta_dataframe.csv", index=False)

alpha_calibrated = np.clip((1.0 - gamma_summary["bar_gamma_overall"]) + gamma_summary["bar_gamma_overall"] * TAU_GRID, 0.0, 1.0)

def curve_df_from_values(
    df: pd.DataFrame,
    value_col: str,
    curve_type: str,
    display_grid: np.ndarray,
    evaluation_grid: np.ndarray,
    grid_kind: str,
) -> pd.DataFrame:
    rows = []
    for sim_name in SIM_ORDER:
        vals = df.loc[df["sim"] == sim_name, value_col].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            continue
        curve_vals = empirical_quantile_curve(vals, evaluation_grid)
        for tau, eval_alpha, curve_val in zip(display_grid, evaluation_grid, curve_vals):
            rows.append(
                {
                    "curve_type": curve_type,
                    "sim": sim_name,
                    "quantile_level": float(tau),
                    "tau": float(tau),
                    "evaluation_quantile_level": float(eval_alpha),
                    "curve_value": float(curve_val),
                    "grid_kind": grid_kind,
                }
            )
    return pd.DataFrame(rows)

calibrated_curve_df = curve_df_from_values(
    calibrated_delta_df,
    value_col="delta",
    curve_type="calibrated",
    display_grid=TAU_GRID,
    evaluation_grid=alpha_calibrated,
    grid_kind="theorem_shifted_alpha",
)
plugin_X_curve_df = curve_df_from_values(
    plugin_X_base,
    value_col="plugin_delta",
    curve_type="plugin_X",
    display_grid=ALPHA_RAW_GRID,
    evaluation_grid=ALPHA_RAW_GRID,
    grid_kind="raw_alpha_full",
)
plugin_XQ_curve_df = curve_df_from_values(
    plugin_XQ_base,
    value_col="plugin_delta",
    curve_type="plugin_XQ",
    display_grid=ALPHA_RAW_GRID,
    evaluation_grid=ALPHA_RAW_GRID,
    grid_kind="raw_alpha_full",
)
oracle_curve_df = curve_df_from_values(
    oracle_emp_base,
    value_col="oracle_emp_delta",
    curve_type="oracle_empirical",
    display_grid=ALPHA_RAW_GRID,
    evaluation_grid=ALPHA_RAW_GRID,
    grid_kind="raw_alpha_full",
)
plugin_X_common_df = curve_df_from_values(
    plugin_X_base,
    value_col="plugin_delta",
    curve_type="plugin_X",
    display_grid=TAU_GRID,
    evaluation_grid=TAU_GRID,
    grid_kind="common_calibrated_window",
)
plugin_XQ_common_df = curve_df_from_values(
    plugin_XQ_base,
    value_col="plugin_delta",
    curve_type="plugin_XQ",
    display_grid=TAU_GRID,
    evaluation_grid=TAU_GRID,
    grid_kind="common_calibrated_window",
)

curve_df = pd.concat(
    [
        calibrated_curve_df,
        oracle_curve_df,
        plugin_X_curve_df,
        plugin_XQ_curve_df,
        plugin_X_common_df,
        plugin_XQ_common_df,
    ],
    ignore_index=True,
)
curve_df.to_csv(OUTPUT_DIR / "curve_dataframe.csv", index=False)

for sim_name in SIM_ORDER:
    sub_x = plugin_X_base.loc[plugin_X_base["sim"] == sim_name, "plugin_delta"].to_numpy(dtype=float)
    sub_xq = plugin_XQ_base.loc[plugin_XQ_base["sim"] == sim_name, "plugin_delta"].to_numpy(dtype=float)
    if np.nanstd(sub_x) <= 1e-12:
        warnings.warn(f"plugin_X discrepancy is nearly constant for {sim_name}.")
    if np.nanstd(sub_xq) <= 1e-12:
        warnings.warn(f"plugin_XQ discrepancy is nearly constant for {sim_name}.")

display(plugin_delta_df.head())
display(curve_df.head(12))


### Figures

Interpretation note

- Solid curves are the calibrated, confidence-set-based curves.
- Dashed / alternate-style curves are predictive plug-in references only.
- The x-axis is `tau`. For calibrated curves, `tau` is mapped internally to the shifted quantile index from the theorem; for predictive curves, `tau` is the raw empirical quantile level.
- A predictive curve lying above a calibrated curve is not automatically a bug; the predictive model may simply be weak over only ~235 questions.


In [ ]:
# Main figure: calibrated vs simulator-assisted plugin_XQ
fig, ax = plt.subplots(figsize=(12, 7))
for sim_name in SIM_ORDER:
    color = SIM_COLOR_MAP.get(sim_name)
    cal = calibrated_curve_df[calibrated_curve_df["sim"] == sim_name].sort_values("quantile_level")
    plug = plugin_XQ_curve_df[plugin_XQ_curve_df["sim"] == sim_name].sort_values("quantile_level")
    if not cal.empty:
        ax.plot(cal["quantile_level"], cal["curve_value"], color=color, linestyle="-", linewidth=2.8, label=f"{sim_name} calibrated")
    if not plug.empty:
        ax.plot(plug["quantile_level"], plug["curve_value"], color=color, linestyle="--", linewidth=2.4, label=f"{sim_name} plugin_XQ")
ax.set_xlabel(r"$\tau$")
ax.set_ylabel("Quantile curve value")
ax.set_title("Calibrated upper curve vs full-data simulator-assisted predictive benchmark")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
ax.grid(True, alpha=0.25)
plt.tight_layout()
main_plot_path = OUTPUT_DIR / "worldvalue_full_data_calibrated_vs_plugin_XQ_overlay.png"
main_archive_path = FIGURES_DIR / "worldvalue_benchmark_calibrated_vs_simulator_assisted_plugin.png"
fig.savefig(main_plot_path, dpi=220, bbox_inches="tight")
fig.savefig(main_archive_path, dpi=220, bbox_inches="tight")
plt.show()

# Secondary figure: calibrated vs question-only plugin_X
fig, ax = plt.subplots(figsize=(12, 7))
for sim_name in SIM_ORDER:
    color = SIM_COLOR_MAP.get(sim_name)
    cal = calibrated_curve_df[calibrated_curve_df["sim"] == sim_name].sort_values("quantile_level")
    plug = plugin_X_curve_df[plugin_X_curve_df["sim"] == sim_name].sort_values("quantile_level")
    if not cal.empty:
        ax.plot(cal["quantile_level"], cal["curve_value"], color=color, linestyle="-", linewidth=2.8, label=f"{sim_name} calibrated")
    if not plug.empty:
        ax.plot(plug["quantile_level"], plug["curve_value"], color=color, linestyle="--", linewidth=2.4, label=f"{sim_name} plugin_X")
ax.set_xlabel(r"$\tau$")
ax.set_ylabel("Quantile curve value")
ax.set_title("Calibrated upper curve vs full-data question-only predictive benchmark")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
ax.grid(True, alpha=0.25)
plt.tight_layout()
secondary_plot_path = OUTPUT_DIR / "worldvalue_full_data_calibrated_vs_plugin_X_overlay.png"
secondary_archive_path = FIGURES_DIR / "worldvalue_benchmark_calibrated_vs_question_only_plugin.png"
fig.savefig(secondary_plot_path, dpi=220, bbox_inches="tight")
fig.savefig(secondary_archive_path, dpi=220, bbox_inches="tight")
plt.show()

# Optional direct benchmark comparison: plugin_X vs plugin_XQ
fig, ax = plt.subplots(figsize=(12, 7))
for sim_name in SIM_ORDER:
    color = SIM_COLOR_MAP.get(sim_name)
    plug_x = plugin_X_curve_df[plugin_X_curve_df["sim"] == sim_name].sort_values("quantile_level")
    plug_xq = plugin_XQ_curve_df[plugin_XQ_curve_df["sim"] == sim_name].sort_values("quantile_level")
    if not plug_x.empty:
        ax.plot(plug_x["quantile_level"], plug_x["curve_value"], color=color, linestyle=":", linewidth=2.2, label=f"{sim_name} plugin_X")
    if not plug_xq.empty:
        ax.plot(plug_xq["quantile_level"], plug_xq["curve_value"], color=color, linestyle="--", linewidth=2.4, label=f"{sim_name} plugin_XQ")
ax.set_xlabel(r"$\tau$")
ax.set_ylabel("Quantile curve value")
ax.set_title("Full-data question-only vs simulator-assisted predictive benchmarks")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
ax.grid(True, alpha=0.25)
plt.tight_layout()
benchmark_compare_plot_path = OUTPUT_DIR / "worldvalue_full_data_plugin_X_vs_plugin_XQ.png"
benchmark_compare_archive_path = FIGURES_DIR / "worldvalue_benchmark_question_only_vs_simulator_assisted_plugin.png"
fig.savefig(benchmark_compare_plot_path, dpi=220, bbox_inches="tight")
fig.savefig(benchmark_compare_archive_path, dpi=220, bbox_inches="tight")
plt.show()

print("Saved figures")
for path in [
    main_plot_path,
    secondary_plot_path,
    benchmark_compare_plot_path,
    main_archive_path,
    secondary_archive_path,
    benchmark_compare_archive_path,
]:
    print(f"- {path.relative_to(REPRO_ROOT)}")


### Post-figure benchmark diagnostics

Interpretation guide

- `calibrated`: the adjusted upper curve produced by the confidence-set construction
- `oracle_empirical`: the empirical quantile curve of `loss(p_full, qhat)` using the strongest full-data human target available
- `plugin_X`, `plugin_XQ`: empirical quantile curves of `loss(p_tilde, qhat)` using predicted human targets

The predictive plug-in curves are intended as empirical approximations to the oracle empirical curve. Because they use predicted targets `p_tilde` instead of the full-data target `p_full`, they can lie above or below the oracle empirical curve depending on prediction error.

Crossing guidance

- Crossings between calibrated and predictive curves do not automatically imply a bug.
- Crossings can arise because the plug-in curves use predicted `p_tilde`, because empirical quantiles fluctuate in finite samples, because `plugin_X` and `plugin_XQ` are different predictors, and because `plugin_XQ` conditions on `qhat`.
- `plugin_XQ` can look much more favorable than `plugin_X` because it uses the same simulator-side `qhat` that also appears in the final loss.
- Upper-tail crossings may be driven by a small number of hard questions.
- Broad or systematic crossings should still trigger implementation checks, which is why the oracle empirical curve is included below as the main diagnostic reference.

Simulator note

- Cases like GPT4o can legitimately show `plugin_X` above calibrated while `plugin_XQ` is below calibrated: the shared question-only predictor may be weaker than GPT4o's own `qhat`, while the simulator-assisted predictor can track that simulator much more closely by conditioning on `qhat`.


In [ ]:
def auc_from_curve(alpha, values):
    return float(np.trapz(np.asarray(values, float), np.asarray(alpha, float)))

# Strong sanity checks before interpreting crossings
if question_df["qid"].nunique() != len(question_df):
    raise ValueError("question_df does not have unique qids.")
if simulator_qhat_df[["qid", "sim"]].duplicated().any():
    dupes = simulator_qhat_df.loc[simulator_qhat_df[["qid", "sim"]].duplicated(), ["qid", "sim"]].head()
    raise ValueError(f"simulator_qhat_df has duplicate (qid, sim) rows: {dupes.to_dict('records')}")
if pred_XQ_df[["qid", "sim"]].duplicated().any():
    dupes = pred_XQ_df.loc[pred_XQ_df[["qid", "sim"]].duplicated(), ["qid", "sim"]].head()
    raise ValueError(f"pred_XQ_df has duplicate (qid, sim) rows: {dupes.to_dict('records')}")

qids_question = set(question_df["qid"])
qids_target = set(selected_target_df.loc[selected_target_df["human_target"].notna(), "qid"])
qids_plugin_x = set(pred_X_df["qid"])
qids_qhat = set(simulator_qhat_df["qid"])
if not (qids_question == qids_target == qids_plugin_x == qids_qhat):
    raise ValueError(
        "Perfect qid alignment failed across question_df, selected_target_df, pred_X_df, and simulator_qhat_df."
    )

if not np.isfinite(question_df["human_target"]).all():
    raise ValueError("Non-finite values found in the selected full-data human target.")
if not np.isfinite(simulator_qhat_df["qhat"]).all():
    raise ValueError("Non-finite values found in simulator qhat values.")

for obj_name, arr in {
    "human_target": question_df["human_target"].to_numpy(float),
    "qhat": simulator_qhat_df["qhat"].to_numpy(float),
    "plugin_X_delta": plugin_X_base["plugin_delta"].to_numpy(float),
    "plugin_XQ_delta": plugin_XQ_base["plugin_delta"].to_numpy(float),
    "oracle_emp_delta": oracle_emp_base["oracle_emp_delta"].to_numpy(float),
    "calibrated_delta": calibrated_delta_df["delta"].to_numpy(float),
}.items():
    if not np.isfinite(arr).all():
        raise ValueError(f"Non-finite curve inputs found in {obj_name}.")

if (question_df["human_target"].min() < -1.000001) or (question_df["human_target"].max() > 1.000001):
    raise ValueError("Selected full-data human target is outside the expected [-1, 1] scale.")
if (simulator_qhat_df["qhat"].min() < -1.000001) or (simulator_qhat_df["qhat"].max() > 1.000001):
    raise ValueError("Simulator qhat is outside the expected [-1, 1] scale.")

oracle_comparison_curve_df = pd.concat(
    [
        calibrated_curve_df,
        oracle_curve_df,
        plugin_X_curve_df,
        plugin_XQ_curve_df,
    ],
    ignore_index=True,
)
oracle_comparison_curve_df.to_csv(OUTPUT_DIR / "oracle_comparison_curve_dataframe.csv", index=False)

oracle_crossing_rows = []
for sim_name in SIM_ORDER:
    cal = calibrated_curve_df[calibrated_curve_df["sim"] == sim_name][["quantile_level", "curve_value"]].rename(columns={"curve_value": "calibrated"})
    oracle = oracle_curve_df[oracle_curve_df["sim"] == sim_name][["quantile_level", "curve_value"]].rename(columns={"curve_value": "oracle_empirical"})
    px = plugin_X_curve_df[plugin_X_curve_df["sim"] == sim_name][["quantile_level", "curve_value"]].rename(columns={"curve_value": "plugin_X"})
    pxq = plugin_XQ_curve_df[plugin_XQ_curve_df["sim"] == sim_name][["quantile_level", "curve_value"]].rename(columns={"curve_value": "plugin_XQ"})
    merged = cal.merge(oracle, on="quantile_level", how="inner").merge(px, on="quantile_level", how="inner").merge(pxq, on="quantile_level", how="inner")
    if len(merged) != len(TAU_GRID):
        raise ValueError(f"Curve alignment failed for {sim_name}.")

    oracle_crossing_rows.append(
        {
            "sim": sim_name,
            "auc_calibrated": auc_from_curve(merged["quantile_level"], merged["calibrated"]),
            "auc_oracle_empirical": auc_from_curve(merged["quantile_level"], merged["oracle_empirical"]),
            "auc_plugin_X": auc_from_curve(merged["quantile_level"], merged["plugin_X"]),
            "auc_plugin_XQ": auc_from_curve(merged["quantile_level"], merged["plugin_XQ"]),
            "frac_oracle_empirical_gt_calibrated": float(np.mean(merged["oracle_empirical"] > merged["calibrated"])),
            "frac_plugin_X_gt_calibrated": float(np.mean(merged["plugin_X"] > merged["calibrated"])),
            "frac_plugin_XQ_gt_calibrated": float(np.mean(merged["plugin_XQ"] > merged["calibrated"])),
            "frac_plugin_X_gt_oracle_empirical": float(np.mean(merged["plugin_X"] > merged["oracle_empirical"])),
            "frac_plugin_XQ_gt_oracle_empirical": float(np.mean(merged["plugin_XQ"] > merged["oracle_empirical"])),
        }
    )

oracle_crossing_summary_df = pd.DataFrame(oracle_crossing_rows)
oracle_crossing_summary_df["suspicious_oracle_gt_calibrated_broad"] = oracle_crossing_summary_df["frac_oracle_empirical_gt_calibrated"] > 0.5
oracle_crossing_summary_df.to_csv(OUTPUT_DIR / "oracle_crossing_summary.csv", index=False)

mean_oracle_gt_cal = float(oracle_crossing_summary_df["frac_oracle_empirical_gt_calibrated"].mean())
mean_plugin_x_gt_oracle = float(oracle_crossing_summary_df["frac_plugin_X_gt_oracle_empirical"].mean())
mean_plugin_xq_gt_oracle = float(oracle_crossing_summary_df["frac_plugin_XQ_gt_oracle_empirical"].mean())
suspicious_count = int(oracle_crossing_summary_df["suspicious_oracle_gt_calibrated_broad"].sum())

if mean_oracle_gt_cal < 0.5:
    oracle_vs_cal_message = "The calibrated curve is mostly above the oracle empirical curve, which is the expected qualitative pattern."
else:
    oracle_vs_cal_message = "The oracle empirical curve is broadly above calibrated, which suggests an indexing or implementation mismatch and should be investigated."

if mean_plugin_x_gt_oracle > 0.5:
    plugin_x_message = "The question-only plug-in curve is mostly above the oracle empirical curve, which is consistent with prediction error in the shared semantic benchmark."
else:
    plugin_x_message = "The question-only plug-in curve is mostly below the oracle empirical curve."

if mean_plugin_xq_gt_oracle > 0.5:
    plugin_xq_message = "The simulator-assisted plug-in curve is mostly above the oracle empirical curve."
else:
    plugin_xq_message = "The simulator-assisted plug-in curve is mostly below the oracle empirical curve; this can happen because it conditions on qhat and may therefore be especially favorable."

suspicious_message = (
    "No suspicious broad oracle-vs-calibrated crossing pattern was detected."
    if suspicious_count == 0
    else f"{suspicious_count} simulator(s) show a broad oracle empirical crossing above calibrated and should be checked more closely."
)

display(Markdown("### Oracle comparison summary"))
display(oracle_crossing_summary_df)
display(
    Markdown(
        "\n".join(
            [
                "### Concise conclusion",
                f"- {oracle_vs_cal_message}",
                f"- {plugin_x_message}",
                f"- {plugin_xq_message}",
                f"- {suspicious_message}",
            ]
        )
    )
)


### Diagnostics and summary tables

These checks are here to distinguish genuine predictive weakness from alignment or scaling bugs.


In [ ]:
def auc_from_curve(alpha, values):
    return float(np.trapz(np.asarray(values, float), np.asarray(alpha, float)))

def cvar_like_from_curve(alpha, values, alpha0=0.90):
    alpha = np.asarray(alpha, float)
    values = np.asarray(values, float)
    mask = alpha >= alpha0
    if mask.sum() < 2:
        return np.nan
    return float(np.trapz(values[mask], alpha[mask]) / max(1e-12, (alpha[mask][-1] - alpha[mask][0])))

alignment_checks = pd.DataFrame(
    [
        {"check": "retained_qids", "n_unique": len(retained_questions)},
        {"check": "question_df_qids", "n_unique": int(question_df["qid"].nunique())},
        {"check": "selected_human_target_qids", "n_unique": int(question_df["human_target"].notna().sum())},
        {"check": "common_qids_calibrated", "n_unique": len(common_qids)},
        {"check": "simulator_qhat_qids", "n_unique": int(simulator_qhat_df["qid"].nunique())},
        {"check": "plugin_X_qids", "n_unique": int(pred_X_df["qid"].nunique())},
        {"check": "plugin_XQ_qids", "n_unique": int(pred_XQ_df["qid"].nunique())},
        {"check": "qid_sets_all_match", "n_unique": int(
            (set(retained_questions) == set(question_df["qid"]) == set(common_qids) == set(simulator_qhat_df["qid"]))
        )},
    ]
)

scale_checks_df = pd.DataFrame(
    [
        {
            "object": "human_target",
            "min": float(np.nanmin(question_df["human_target"])),
            "max": float(np.nanmax(question_df["human_target"])),
            "mean": float(np.nanmean(question_df["human_target"])),
        },
        {
            "object": "qhat_all_sims",
            "min": float(np.nanmin(simulator_qhat_df["qhat"])),
            "max": float(np.nanmax(simulator_qhat_df["qhat"])),
            "mean": float(np.nanmean(simulator_qhat_df["qhat"])),
        },
    ]
)

predictor_category_df = (
    pred_X_df.groupby("category", dropna=False)
    .agg(
        n_qids=("qid", "count"),
        rmse=("sq_error", lambda s: float(np.sqrt(np.mean(s)))),
        mae=("abs_error", "mean"),
    )
    .reset_index()
    .sort_values(["rmse", "n_qids"], ascending=[False, False])
)
predictor_category_df.to_csv(OUTPUT_DIR / "predictor_category_summary.csv", index=False)

discrepancy_summary_df = (
    plugin_delta_df.groupby(["curve_type", "sim"], dropna=False)
    .agg(
        n_qids=("qid", "nunique"),
        mean_plugin_delta=("plugin_delta", "mean"),
        median_plugin_delta=("plugin_delta", "median"),
        mean_oracle_sq_loss=("oracle_sq_loss", "mean"),
        missing_qhat_n=("qhat", lambda s: int(s.isna().sum())),
        missing_pred_n=("human_target_pred_oof", lambda s: int(s.isna().sum())),
    )
    .reset_index()
    .sort_values(["curve_type", "sim"])
)
discrepancy_summary_df.to_csv(OUTPUT_DIR / "discrepancy_summary.csv", index=False)

curve_summary_rows = []
for sim_name in SIM_ORDER:
    cal = calibrated_curve_df[calibrated_curve_df["sim"] == sim_name].sort_values("quantile_level")
    oracle = oracle_curve_df[oracle_curve_df["sim"] == sim_name].sort_values("quantile_level")
    x_common = plugin_X_common_df[plugin_X_common_df["sim"] == sim_name].sort_values("quantile_level")
    xq_common = plugin_XQ_common_df[plugin_XQ_common_df["sim"] == sim_name].sort_values("quantile_level")
    x_full = plugin_X_curve_df[plugin_X_curve_df["sim"] == sim_name].sort_values("quantile_level")
    xq_full = plugin_XQ_curve_df[plugin_XQ_curve_df["sim"] == sim_name].sort_values("quantile_level")
    if cal.empty or oracle.empty or x_common.empty or xq_common.empty:
        continue
    curve_summary_rows.append(
        {
            "sim": sim_name,
            "human_target_source": selected_target_source,
            "embedding_source": embedding_source,
            "bar_gamma_overall": float(gamma_summary["bar_gamma_overall"]),
            "alpha_min_calibrated": float(alpha_calibrated.min()),
            "auc_calibrated_shifted_alpha": auc_from_curve(cal["quantile_level"], cal["curve_value"]),
            "auc_oracle_empirical": auc_from_curve(oracle["quantile_level"], oracle["curve_value"]),
            "auc_plugin_X_common_window": auc_from_curve(x_common["quantile_level"], x_common["curve_value"]),
            "auc_plugin_XQ_common_window": auc_from_curve(xq_common["quantile_level"], xq_common["curve_value"]),
            "auc_plugin_X_full_alpha": auc_from_curve(x_full["quantile_level"], x_full["curve_value"]),
            "auc_plugin_XQ_full_alpha": auc_from_curve(xq_full["quantile_level"], xq_full["curve_value"]),
            "cvar_calibrated_shifted_alpha": cvar_like_from_curve(cal["quantile_level"], cal["curve_value"], alpha0=TAIL_ALPHA0),
            "cvar_oracle_empirical": cvar_like_from_curve(oracle["quantile_level"], oracle["curve_value"], alpha0=TAIL_ALPHA0),
            "cvar_plugin_X_common_window": cvar_like_from_curve(x_common["quantile_level"], x_common["curve_value"], alpha0=TAIL_ALPHA0),
            "cvar_plugin_XQ_common_window": cvar_like_from_curve(xq_common["quantile_level"], xq_common["curve_value"], alpha0=TAIL_ALPHA0),
        }
    )

curve_summary_df = pd.DataFrame(curve_summary_rows)
curve_summary_df.to_csv(OUTPUT_DIR / "curve_summary.csv", index=False)

display(Markdown("### Alignment checks"))
display(alignment_checks)
display(Markdown("### Dataset selection"))
display(target_selection_df)
display(full_target_consistency_summary)
display(Markdown("### Scale checks"))
display(scale_checks_df)
display(Markdown("### Predictor summary"))
display(predictor_summary_df)
display(Markdown("### Discrepancy summary"))
display(discrepancy_summary_df)
display(Markdown("### Curve summary"))
display(curve_summary_df)

print("Saved artifacts")
for path in [
    OUTPUT_DIR / "question_feature_dataframe.csv",
    OUTPUT_DIR / "oof_prediction_question_only.csv",
    OUTPUT_DIR / "oof_prediction_simulator_assisted.csv",
    OUTPUT_DIR / "human_target_source_summary.csv",
    OUTPUT_DIR / "human_target_consistency_summary.csv",
    OUTPUT_DIR / "predictor_summary.csv",
    OUTPUT_DIR / "predictor_category_summary.csv",
    OUTPUT_DIR / "crossfit_fold_selection_question_only.csv",
    OUTPUT_DIR / "crossfit_fold_selection_simulator_assisted.csv",
    OUTPUT_DIR / "calibrated_delta_dataframe.csv",
    OUTPUT_DIR / "simulator_qhat_dataframe.csv",
    OUTPUT_DIR / "plugin_delta_dataframe.csv",
    OUTPUT_DIR / "oracle_empirical_delta_dataframe.csv",
    OUTPUT_DIR / "curve_dataframe.csv",
    OUTPUT_DIR / "oracle_comparison_curve_dataframe.csv",
    OUTPUT_DIR / "curve_summary.csv",
    OUTPUT_DIR / "discrepancy_summary.csv",
    OUTPUT_DIR / "oracle_crossing_summary.csv",
    main_plot_path,
    secondary_plot_path,
    benchmark_compare_plot_path,
]:
    print(f"- {path.relative_to(REPRO_ROOT)}")
